# Guangzhou LST + NDVI Calculation

In [ ]:
# ==========================================
# STEP 1: INITIALIZATION & DEPENDENCIES
# ==========================================
# !pip install geemap ee  # Uncomment if you need to install them first

import ee
import geemap

# Initialize the Earth Engine library
try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

In [ ]:
# ==========================================
# STEP 2: DEFINE COUPLING & MASKING FUNCTIONS
# ==========================================

# 1. Define the Area of Interest (AOI) bounding box around Guangzhou
guangzhou_aoi = ee.Geometry.Rectangle([113.13, 22.85, 113.7, 23.5])

# 2. Cloud masking function using the QA_PIXEL band
def mask_landsat_clouds(image):
    # Bits 3 and 4 correspond to Cloud and Cloud Shadow respectively
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)

    # Combine masks
    mask = cloud_shadow_mask.And(cloud_mask)
    return image.updateMask(mask)

# 3. Scale Thermal Band 10 to Kelvin and convert directly to Celsius
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    lst_celsius = lst_celsius.rename('LST_Celsius')
    return image.addBands(optical_bands, None, True).addBands(lst_celsius, None, True)

# 4. Normalize LST by subtracting the per-image spatial mean
def normalize_lst(image):
    lst = image.select('LST_Celsius')

    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=guangzhou_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')

    return (
        lst.subtract(image_mean)
           .rename('LST_Normalized')
           .toFloat()
           .copyProperties(image, image.propertyNames())
    )

In [12]:
# ==========================================
# STEP 1: INITIALIZATION & DEPENDENCIES
# ==========================================
# !pip install geemap ee  # Uncomment if you need to install them first

import ee
import geemap

# Initialize the Earth Engine library
try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# ==========================================
# STEP 2: DEFINE COUPLING & MASKING FUNCTIONS
# ==========================================

# 1. Define the Area of Interest (AOI) bounding box around Guangzhou
guangzhou_aoi = ee.Geometry.Rectangle([113.13, 22.85, 113.7, 23.5])

# 2. Cloud masking function using the QA_PIXEL band
def mask_landsat_clouds(image):
    # Bits 3 and 4 correspond to Cloud and Cloud Shadow respectively
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)

    # Combine masks
    mask = cloud_shadow_mask.And(cloud_mask)
    return image.updateMask(mask)

# 3. Scale Thermal Band 10 to Kelvin and convert directly to Celsius
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    lst_celsius = lst_celsius.rename('LST_Celsius')
    return image.addBands(optical_bands, None, True).addBands(lst_celsius, None, True)

# 4. Normalize LST by subtracting the per-image spatial mean
def normalize_lst(image):
    lst = image.select('LST_Celsius')

    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=guangzhou_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')

    return (
        lst.subtract(image_mean)
           .rename('LST_Normalized')
           .toFloat()
           .copyProperties(image, image.propertyNames())
    )

# ==========================================
# STEP 3: FILTER COLLECTION & PROCESS MEDIAN
# ==========================================

def build_collection(start_date, end_date):
    return (
        ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        .filterBounds(guangzhou_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(mask_landsat_clouds)
        .map(apply_scale_factors)
        .map(normalize_lst)
    )

landsat_2023 = build_collection('2023-05-01', '2023-08-31')
landsat_2024 = build_collection('2024-05-01', '2024-08-31')
landsat_2025 = build_collection('2025-05-01', '2025-08-31')

landsat_collection = landsat_2023.merge(landsat_2024).merge(landsat_2025)

lst_composite = landsat_collection.select('LST_Normalized').mean().clip(guangzhou_aoi)

print(f"Images found - 2023: {landsat_2023.size().getInfo()}, 2024: {landsat_2024.size().getInfo()}, 2025: {landsat_2025.size().getInfo()}, total: {landsat_collection.size().getInfo()}")

# ==========================================
# STEP 4: INTERACTIVE VISUALIZATION
# ==========================================

Map = geemap.Map(center=[23.1291, 113.2644], zoom=11)

# Visualization parameters updated for normalized data:
# Values now represent deviation from each image's mean (centred around 0)
lst_vis = {
    'min': -10.0,  # 10°C below the image mean
    'max': 10.0,   # 10°C above the image mean
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}

Map.addLayer(lst_composite, lst_vis, 'Guangzhou Summer LST 2023-2025 (Normalized)')
Map.add_colorbar(lst_vis, label="LST Anomaly (°C relative to image mean)")

# ==========================================
# STEP 5: EXPORT TO LOCAL PROCESSED DATA
# ==========================================

print("Exporting Guangzhou LST map locally...")
try:
    geemap.ee_export_image(
        lst_composite,
        filename='../data/processed/guangzhou_lst_2023_2025.tif',
        scale=30,
        region=guangzhou_aoi,
        file_per_band=False
    )
except Exception as e:
    print(e)

Images found - 2023: 2, 2024: 1, 2025: 2, total: 5
Exporting Guangzhou LST map locally...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\processed\guangzhou_lst_2023_2025.tif


In [14]:
import rasterio
import numpy as np

with rasterio.open("../data/processed/guangzhou_lst_2023_2025.tif") as src:
    data = src.read(1)
    profile = src.profile

with rasterio.open(
    "../data/processed/guangzhou_lst_2023_2025_clean.tif",
    "w",
    **profile
) as dst:
    dst.write(data, 1)

# NDVI Calculation

In [16]:
# ==========================================
# STEP 1: SPATIAL BOUNDS & IMAGE PROCESSING
# ==========================================

# 1. Landsat 8 Cloud and Shadow Masking Function
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cloud_shadow_mask.And(cloud_mask))

# 2. Combined Scaling, LST, and NDVI Calculation Function
def process_landsat_indicators(image):
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST_Celsius')
    return image.addBands(optical, None, True).addBands(lst_celsius, None, True).addBands(ndvi)

# 3. Normalize LST by subtracting the per-image spatial mean
def normalize_lst(image):
    lst = image.select('LST_Celsius')
    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=guangzhou_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')
    return lst.subtract(image_mean).rename('LST_Normalized').toFloat().copyProperties(image, image.propertyNames())

# ==========================================
# STEP 2: DATA ENGINE AGGREGATION
# ==========================================

def build_collection(start_date, end_date):
    return (
        ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        .filterBounds(guangzhou_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(mask_landsat_clouds)
        .map(process_landsat_indicators)
    )

landsat_2023 = build_collection('2023-05-01', '2023-08-31')
landsat_2024 = build_collection('2024-05-01', '2024-08-31')
landsat_2025 = build_collection('2025-05-01', '2025-08-31')

landsat_collection = landsat_2023.merge(landsat_2024).merge(landsat_2025)

# Normalize LST per-image before taking the median
lst_normalized = landsat_collection.map(normalize_lst)

composite = ee.Image.cat([
    lst_normalized.select('LST_Normalized').median(),
    landsat_collection.select('NDVI').median()
]).clip(guangzhou_aoi)

lst_layer  = composite.select('LST_Normalized')
ndvi_layer = composite.select('NDVI')

print(f"Images found - 2023: {landsat_2023.size().getInfo()}, 2024: {landsat_2024.size().getInfo()}, 2025: {landsat_2025.size().getInfo()}, total: {landsat_collection.size().getInfo()}")

# ==========================================
# STEP 3: INTERACTIVE TARGETED DISPLAY
# ==========================================

Map = geemap.Map(center=[23.1291, 113.2644], zoom=11)

lst_vis = {
    'min': -10.0, 'max': 10.0,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}

ndvi_vis = {
    'min': -0.3, 'max': 0.3,
    'palette': ['#CE7E45', '#DF923D', '#F1B555', '#FCD163', '#99B718', '#74A028', '#3E861A', '#206E1A', '#053C1A']
}

Map.addLayer(ndvi_layer, ndvi_vis, 'Guangzhou Summer NDVI 2023-2025 (Normalized)')
Map.addLayer(lst_layer,  lst_vis,  'Guangzhou Summer LST 2023-2025 (Normalized)')
Map.add_colorbar(lst_vis,  label="LST Anomaly (°C relative to image mean)")
Map.add_colorbar(ndvi_vis, label="NDVI Anomaly (relative to image mean)")

display(Map)

# ==========================================
# STEP 4: FILE EXPORT TO CURRENT DIR STRUCTURE
# ==========================================
print("Saving output rasters to local directories...")

geemap.ee_export_image(
    lst_layer,
    filename='../data/processed/guangzhou_lst_2023_2025.tif',
    scale=30, region=guangzhou_aoi, file_per_band=False
)

geemap.ee_export_image(
    ndvi_layer,
    filename='../data/processed/guangzhou_ndvi_2023_2025.tif',
    scale=30, region=guangzhou_aoi, file_per_band=False
)
print("Done! Check your data/processed/ folder.")

Images found - 2023: 2, 2024: 1, 2025: 2, total: 5


Map(center=[23.1291, 113.2644], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

Saving output rasters to local directories...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\processed\guangzhou_lst_2023_2025.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\processed\guangzhou_ndvi_2023_2025.tif
Done! Check your data/processed/ folder.


## Clean metadata

In [ ]:
import ee
import geemap
import os

print("--- Downloading Guangzhou LCZ Raster from WUDAPT/GEE ---")

# Define Guangzhou bounding box (same envelope as LST/NDVI tiles)
aoi = ee.Geometry.Rectangle([112.8500, 22.5000, 114.1500, 23.9500])

# WUDAPT Local Climate Zones — global 100m dataset on GEE
lcz_collection = ee.ImageCollection("RUB/RUBCLIM/LCZ/global_lcz_map/v1")

# Get the most recent annual composite and clip to Guangzhou
lcz_image = (
    lcz_collection
    .filterBounds(aoi)
    .sort("system:time_start", False)  # Most recent first
    .first()
    .select("LCZ_Filter")              # Use the filtered (cleaned) band, not raw
    .clip(aoi)
)

os.makedirs("../../data/processed", exist_ok=True)
output_path = "../../data/processed/guangzhou_lcz_2018.tif"

print("Exporting LCZ raster (100m resolution)...")
geemap.ee_export_image(
    lcz_image.toFloat(),
    filename=output_path,
    scale=100,           # WUDAPT native resolution
    region=aoi,
    file_per_band=False
)

print(f"✔ LCZ raster saved to {output_path}")

In [ ]:
%pip install osmnx

In [ ]:
import geopandas as gpd
import osmnx as ox
import requests
import os
os.makedirs("../../data/guangzhou", exist_ok=True)

# ============================================================
# FIX: Download proper Guangzhou subdistrict polygons from OSM
# (Replaces the original boundary-only guangzhou_admin.geojson
#  which contained only LineStrings with empty properties)
# ============================================================

print("Downloading Guangzhou admin boundaries from OSM...")

# Get the official Guangzhou city boundary as a clipping mask
guangzhou_boundary = ox.geocode_to_gdf("Guangzhou, Guangdong, China")
guangzhou_boundary = guangzhou_boundary.to_crs(epsg=4326)

# Download admin level 9 subdistricts (街道 / Jiedao level)
gdf_gz = ox.features_from_place(
    "Guangzhou, Guangdong, China",
    tags={"boundary": "administrative", "admin_level": "9"}
)

# Keep only polygon geometries
gdf_gz = gdf_gz[gdf_gz.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
gdf_gz = gdf_gz.reset_index(drop=True)
gdf_gz = gdf_gz.to_crs(epsg=4326)

# Clip strictly to Guangzhou city boundary to remove stray neighbouring districts
gdf_gz = gpd.clip(gdf_gz, guangzhou_boundary)

print(f"Found {len(gdf_gz)} subdistrict units within Guangzhou city boundary")

output = "../../data/Guangzhou/guangzhou_admin.geojson"
gdf_gz[['geometry', 'name']].to_file(output, driver="GeoJSON")
print(f"✔ Saved {len(gdf_gz)} polygon subdistricts to {output}")

In [ ]:
import geopandas as gpd
from rasterstats import zonal_stats

print("\n--- Running Zonal Statistics ---")
geojson_path = "../data/guangzhou/guangzhou_admin.geojson"
output_path  = "../../data/processed/guangzhou_admin_enriched.geojson"

gdf = gpd.read_file(geojson_path)
gdf['geometry'] = gdf['geometry'].make_valid()

# Extract values from our seamlessly stitched mosaics
gdf['mean_LST_celsius'] = [x['mean'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_lst_2025.tif", stats=['mean'])]
gdf['mean_NDVI']        = [x['mean'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_ndvi_2025.tif", stats=['mean'])]
gdf['majority_LCZ']     = [x['majority'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_lcz_2018.tif", stats=['majority'])]

print(f"Missing LST values remaining: {gdf['mean_LST_celsius'].isna().sum()}")
print(f"Missing NDVI values remaining: {gdf['mean_NDVI'].isna().sum()}")

gdf.to_file(output_path, driver="GeoJSON")
print("✔ Enriched dataset saved successfully!")

In [ ]:
import geopandas as gpd

print("--- Final Cleanup: Dropping Edge Thermal Gaps & Water Artifacts ---")
enriched_path = "data/processed/guangzhou_admin_enriched.geojson"

# 1. Load the dataset we just exported
gdf = gpd.read_file(enriched_path)
initial_rows = len(gdf)

# 2. Drop rows missing thermal data
gdf_clean = gdf.dropna(subset=['mean_LST_celsius', 'mean_NDVI'])

# 3. Drop negative NDVI values (water bodies / cloud artifacts)
gdf_clean = gdf_clean[gdf_clean['mean_NDVI'] >= 0]

final_rows = len(gdf_clean)

print(f"-> Total districts before cleanup: {initial_rows}")
print(f"-> Total districts kept with full coverage: {final_rows}")
print(f"-> Successfully removed {initial_rows - final_rows} boundary gap / water rows.")

# 4. Overwrite the GeoJSON with the finalized, clean dataset
gdf_clean.to_file(enriched_path, driver="GeoJSON")
print("✔ Cleaned dataset safely saved! Ready for R clustering.")